## Installing dependencies and setup

In [1]:
#------------------------------------------------
# INSTALLING DEPENDENCIES AND STARTING OLLAMA SERVER
#------------------------------------------------

!nvidia-smi
!apt-get update && apt-get install -y pciutils
!sudo apt-get install zstd 
!pip install deepeval rich pandas tabulate --quiet
!curl -fsSL https://ollama.com/install.sh | sh
!pkill ollama # just to make sure there isn't any existing server running
!pip install ollama

import pathlib
import subprocess
import time
import json
import re
import os
import ollama
from ollama import Client
import httpx
from pydantic import BaseModel
from google.colab import drive

server_process = subprocess.Popen(["ollama", "serve"])

In [2]:
!git clone https://www.github.com/rryyyymmmmmmmm/pfe_llm_eval
!mv pfe_llm_eval/* . && rm -rf pfe_llm_eval

#-----------------------------------------------------------------
# LOADING THE SYSTEM PROMPT
#-----------------------------------------------------------------

prompt = pathlib.Path("system_prompt.txt").read_text(encoding="utf-8")

#-----------------------------------------------------------------
# LOADING THE OCR RESULTS TO BE USED IN THE EXAMPLES AND EVALUATION
#-----------------------------------------------------------------

ordos_ocr = {}

for x in pathlib.Path("ocr_output").glob("*.txt"):
    ordos_ocr[str(x).removeprefix("ocr_output/").removesuffix("_recto_preprocessed.jpg_ocr.txt")] = x.read_text(encoding="utf-8")


# ─────────────────────────────────────────────────────────────
# SYSTEM PROMPT and USER TEMPLATE
# ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
{prompt}
"""
USER_TEMPLATE = """Extract the structured data from this OCR prescription text:
---
{ocr_text}
---"""

In [3]:
#-----------------------------------------------------------------
# LOADING THE EXAMPLES OF FRONT ORDOS TO BE USED IN THE EVALUATION
#-----------------------------------------------------------------

EXAMPLES = []
#!mkdir EXAMPLES

for ex in pathlib.Path("EXAMPLES").glob("*.json"):
    with open(ex, "r", encoding="utf-8") as f:
        data = json.load(f)
        data["ocr_text"] = ordos_ocr[data.get("id")]
        EXAMPLES.append({
            "name": ex.name.removesuffix("_example.json"),
            "data": data
        })

print(EXAMPLES[0].keys())

In [4]:
OLLAMA_BASE_URL = "http://localhost:11434"  
OLLAMA_TIMEOUT  = 300  

#─────────────────────────────────────────────────────────────
# PULLING MODELS
#-─────────────────────────────────────────────────────────────

# from the LLMStructBench paper 
LLM_STRUCT_BENCH_MODELS_BIG = [
    "gemma3:12b",

    "deepseek-r1:14b",
    "deepseek-r1:8b",
    "deepseek-r1:7b",
    
    "phi4:14b",
    "phi3:14b",

    "qwen3:14b",
    "qwen3:8b",
    
]

LLM_STRUCT_BENCH_MODELS_SMALL = [
    "qwen3:4b",
    "qwen3:1.7b",
    "qwen3:0.6b",

    "phi4-mini:3.8b",
    "phi3.5:3.8b",
    "phi3:3.8b",

    "deepseek-r1:1.5b",    
    "gemma3:1b",
]


STRUCT_EVAL_MODELS = [
    "llama3.1:8b-instruct-q4_K_M",
    "llama3:8b-instruct-q4_K_M",
    "phi3:3.8b-mini-128k-instruct-q4_K_M",
    "phi4-mini:3.8b-q4_K_M",
    "qwen2.5:7b-instruct-q4_K_M",
    "qwen3:4b-instruct"
]

# medical models that are popular on Ollama 
MED_MODELS = [
    "medgemma:4b",
    "medgemma:27b"
]

QWEN_MODELS = [
    # Qwen 1
    "qwen:0.5b", 
    "qwen:1.8b",
    "qwen:4b",
    "qwen:7b",

    # Qwen 2
    "qwen2:0.5b", 
    "qwen2:1.5b",
    "qwen2:7b",

    # Qwen 2.5
    "qwen2.5:0.5b",
    "qwen2.5:1.5b",
    "qwen2.5:3b",
    "qwen2.5:7b",

    # Qwen 3
    "qwen3:0.6b",
    "qwen3:1.7b",
    "qwen3:4b",
    "qwen3:4b-instruct",
    "qwen3:8b",
    
    # Qwen 3.5

    # Qwen 3.6
]

#--------------------------------------------------- 
# ASSIGNING WHICH MODELS TO EVAL
#---------------------------------------------------

MODELS_TO_EVAL = LLM_STRUCT_BENCH_MODELS_SMALL

for m in MODELS_TO_EVAL:
    os.system(f"ollama pull {m}")

!ollama list # making sure the models were pulled successfully

client = Client(host=OLLAMA_BASE_URL, timeout=OLLAMA_TIMEOUT) # starting the client to interact with the server

## Evaluation metrics

In [5]:
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase
import math

# ─────────────────────────────────────────────────────────────
# Helper: fuzzy string match (normalise before comparing)
# ─────────────────────────────────────────────────────────────

def normalise(s) -> str: #this function strips all non-alphanumeric characters and lowercases the string for a more accurate comparison
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]", "", str(s).lower()) 

def str_match(a, b) -> bool: #this one just compares two strings for an exact match after normalisation
    return normalise(a) == normalise(b)

def partial_match(pred, gold) -> float: #this function calculates a similarity score between two strings
    """Score 0–1 for fuzzy string similarity."""
    p, g = normalise(pred), normalise(gold)
    if not g:
        return 1.0 if not p else 0.5   # null expected: null predicted = perfect
    if not p:
        return 0.0
    if p == g:
        return 1.0
    # Longest common subsequence ratio
    from difflib import SequenceMatcher
    return SequenceMatcher(None, p, g).ratio()

# ─────────────────────────────────────────────────────────────
# METRIC 1: JSON Validity
# ─────────────────────────────────────────────────────────────
class JSONValidityMetric(BaseMetric): #this is for checking the validity of the JSON output, it's more of an extra check because the model is already supposed to return valid JSON (bcuz of the format="json" parameter in the call), but it's still important to run this test

    """Did the model return parseable JSON at all?"""

    name = "JSON Validity"
    threshold = 1.0

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            parsed = json.loads(test_case.actual_output)
            self.score = 1.0
            self.reason = "Valid JSON"
        except Exception as e:
            self.score = 0.0
            self.reason = f"Invalid JSON: {e}"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


# ─────────────────────────────────────────────────────────────
# METRIC 2: Schema Compliance
# ─────────────────────────────────────────────────────────────
REQUIRED_KEYS = {"medical_facility", "patient", "prescription_date", "doctor", "medications"}

MEDICALFACILITY_STRUCTURED_KEYS = {"name", "type"}
PATIENT_STRUCTURED_KEYS          = {"full_name", "first_name", "family_name", "date_of_birth", "age"}
DOCTOR_STRUCTURED_KEYS           = {"name", "specialty", "license_number"}
MED_STRUCTURED_KEYS              = {"name", "dosage", "form", "posologie", "QSP", "total_quantity", "package_quantity"}


class SchemaComplianceMetric(BaseMetric):
    """Does the JSON match the required schema structure?"""

    name = "Schema Compliance"
    threshold = 0.8

    def _check_nested(self, data: dict, field: str, structured_keys: set) -> bool:
        """Check that a field is a dict with 'raw_text' and a 'structured' sub-dict containing the required keys."""
        obj = data.get(field)
        if not isinstance(obj, dict):
            return False
        if "structured" not in obj:
            return False
        structured = obj["structured"]
        if not isinstance(structured, dict):
            return False
        return structured_keys.issubset(structured.keys())

    def measure(self, test_case: LLMTestCase) -> float:
        # ----------------------------------------------------------
        # JSON parsing
        # ----------------------------------------------------------
        try:
            data = json.loads(test_case.actual_output)
        except Exception:
            self.score = 0.0
            self.reason = "Unparseable JSON"
            return 0.0

        # ----------------------------------------------------------
        # Top-level checks
        # ----------------------------------------------------------
        checks = []
        checks.append(isinstance(data, dict))                        # top-level is an object
        checks.append(REQUIRED_KEYS.issubset(data.keys()))           # all required top-level keys present

        # ----------------------------------------------------------
        # Nested field checks (raw_text + structured sub-keys)
        # ----------------------------------------------------------
        checks.append(self._check_nested(data, "medical_facility", MEDICALFACILITY_STRUCTURED_KEYS))
        checks.append(self._check_nested(data, "patient",          PATIENT_STRUCTURED_KEYS))
        checks.append(self._check_nested(data, "doctor",           DOCTOR_STRUCTURED_KEYS))

        # ----------------------------------------------------------
        # prescription_date: string or null
        # ----------------------------------------------------------
        checks.append(
            "prescription_date" in data and
            (data["prescription_date"] is None or isinstance(data["prescription_date"], str))
        )

        # ----------------------------------------------------------
        # medications: list of dicts, each with raw_text + structured
        # ----------------------------------------------------------
        checks.append(isinstance(data.get("medications"), list))

        meds = data.get("medications", [])
        if meds:
            checks.append(all(
                isinstance(m, dict) and
                "structured" in m and
                isinstance(m["structured"], dict) and
                MED_STRUCTURED_KEYS.issubset(m["structured"].keys())
                for m in meds
            ))

        self.score  = sum(checks) / len(checks)
        self.reason = f"{sum(checks)}/{len(checks)} schema checks passed"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)
    
# ─────────────────────────────────────────────────────────────
# METRIC 3: Entity Accuracy  (the core metric)
# ─────────────────────────────────────────────────────────────
class EntityAccuracyMetric(BaseMetric):
    """
    Compares extracted entities against ground truth.
    expected_output must be the ground truth JSON string.
    """
    name = "Entity Accuracy"
    threshold = 0.7

    def _get_structured(self, data: dict, field: str) -> dict:
        """Safely retrieve the 'structured' sub-dict of a nested field."""
        return data.get(field, {}).get("structured", {})

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            pred = json.loads(test_case.actual_output)
        except Exception:
            self.score = 0.0; self.reason = "Unparseable JSON"; return 0.0

        gold = json.loads(test_case.expected_output)
        scores = []

        # -------------------------------------------------------
        # Patient fields (from structured sub-dict)
        # -------------------------------------------------------
        pred_patient = self._get_structured(pred, "patient")
        gold_patient = self._get_structured(gold, "patient")
        for f in ["full_name", "first_name", "family_name", "date_of_birth", "age"]:
            scores.append(partial_match(pred_patient.get(f), gold_patient.get(f)))

        # -------------------------------------------------------
        # Prescription date (flat field, unchanged)
        # -------------------------------------------------------
        scores.append(partial_match(pred.get("prescription_date"), gold.get("prescription_date")))

        # -------------------------------------------------------
        # Doctor fields (from structured sub-dict)
        # -------------------------------------------------------
        pred_doctor = self._get_structured(pred, "doctor")
        gold_doctor = self._get_structured(gold, "doctor")
        for f in ["name", "specialty", "license_number"]:
            scores.append(partial_match(pred_doctor.get(f), gold_doctor.get(f)))

        # -------------------------------------------------------
        # Medical facility fields (from structured sub-dict)
        # -------------------------------------------------------
        pred_facility = self._get_structured(pred, "medical_facility")
        gold_facility = self._get_structured(gold, "medical_facility")
        for f in ["name", "type"]:
            scores.append(partial_match(pred_facility.get(f), gold_facility.get(f)))

        # -------------------------------------------------------
        # Medication matching (align by index; penalise missing/extra)
        # -------------------------------------------------------
        pred_meds = pred.get("medications", [])
        gold_meds = gold.get("medications", [])
        n = max(len(pred_meds), len(gold_meds), 1)  # avoid div-by-zero when both empty
        med_scores = []
        for i in range(n):
            pm_raw = pred_meds[i] if i < len(pred_meds) else None
            gm_raw = gold_meds[i] if i < len(gold_meds) else None

            pm = (pm_raw or {}).get("structured") or {}   # guard against None value
            gm = (gm_raw or {}).get("structured") or {}   # guard against None value

            for f in ["name", "dosage", "form", "posologie", "QSP"]:
                med_scores.append(partial_match(pm.get(f), gm.get(f)))
        if med_scores:
            scores.append(sum(med_scores) / len(med_scores))

        self.score = round(sum(scores) / len(scores), 4)
        self.reason = f"Mean entity match = {self.score:.2%} over {len(scores)} fields"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)
    
# ─────────────────────────────────────────────────────────────
# METRIC 4: Medication Name Recall
# ─────────────────────────────────────────────────────────────
class MedicationRecallMetric(BaseMetric):
    """How many medication names from the ground truth did the model find?"""
    name = "Medication Recall"
    threshold = 0.8

    def _get_med_name(self, med: dict) -> str:
        return normalise((med or {}).get("structured", {}).get("name"))

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            pred = json.loads(test_case.actual_output)
            gold = json.loads(test_case.expected_output)
        except Exception:
            self.score = 0.0; self.reason = "Parse error"; return 0.0

        gold_names = [self._get_med_name(m) for m in gold.get("medications", [])]
        pred_names = [self._get_med_name(m) for m in pred.get("medications", [])]

        if not gold_names:
            self.score = 1.0; self.reason = "No medications in ground truth"; return 1.0

        found = sum(
            any(partial_match(pn, gn) > 0.8 for pn in pred_names)
            for gn in gold_names
        )
        self.score = round(found / len(gold_names), 4)
        self.reason = f"Found {found}/{len(gold_names)} medications"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)
    
print("✅ All 4 custom metrics defined:")
print("   1. JSONValidityMetric      — did model return parseable JSON?")
print("   2. SchemaComplianceMetric  — does JSON match required schema?")
print("   3. EntityAccuracyMetric    — how accurate are extracted fields?")
print("   4. MedicationRecallMetric  — how many drug names were found?")

## Running the evaluation

In [6]:
CURRENT_TIME = "t" + str(time.localtime().tm_mday) + "-" + str(time.localtime().tm_mon) + "-" + str(time.localtime().tm_year) + "_"+ str(time.localtime().tm_hour) + "-" + str(time.localtime().tm_min)

In [7]:
def check_models_available(models: list) -> dict:
    try:
        # Get the list of models currently on the system
        response = ollama.list()
        print(response)
        available = {m.model for m in response.models}
        
    except Exception as e:
        print(f"⚠️  Could not connect to Ollama or parse list: {e}")
        return {m: False for m in models}

    status = {}
    available_models = []
    for m in models:
        # Check for exact match or if the requested string is part of the tag
        found = m in available or any(m in a for a in available)
        if found: available_models.append(m)
        status[m] = found
        icon = "✅" if found else "❌"
        print(f"  {icon} {m}")
        
    return available_models

def call_ollama(model: str, ocr_text: str, prompt: str, save_output: bool) -> dict:
    
    # loading the prompt template and filling in the OCR text

    user_msg = USER_TEMPLATE.format(ocr_text=ocr_text)
    system_prompt = SYSTEM_PROMPT.format(prompt=prompt)
    t0 = time.time()
    metadata = {
        "model": model,
        "raw_output": "",
        "latency_sec": 0.0,
        "error": None
    }
    parsed_data = None
    
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_msg},
            ],
            format="json", # forces the model to respond in a JSON format
            options={"temperature": 0.0},
        )
        
        metadata["latency_sec"] = time.time() - t0
        raw = response["message"]["content"].strip()
        metadata["raw_output"] = raw

        # Strip markdown fences
        clean = re.sub(r"^```[a-z]*\n?", "", raw)
        clean = re.sub(r"\n?```$", "", clean).strip()

        parsed_data = json.loads(clean)

    except json.JSONDecodeError as e:
        metadata["error"] = f"JSON parse error: {e}"
        metadata["latency_sec"] = time.time() - t0
    except Exception as e:
        metadata["error"] = str(e)
        metadata["latency_sec"] = time.time() - t0

    if save_output:
        # Sanitize model name for filenames (remove special characters)
        safe_name = re.sub(r'[^a-zA-Z0-9]', '_', model)
        
        result_file = f"{safe_name}_result.json"
        metadata_file = f"{safe_name}_metadata.json"
        results_folder_name = f'{CURRENT_TIME}'
        %mkdir -p {results_folder_name}

        %cd {results_folder_name}
    
        with open(result_file, "w", encoding="utf-8") as f:
            json.dump(parsed_data, f, indent=4)
    
        with open(metadata_file, "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=4)
        %cd ..

    return {
        "parsed_json": parsed_data,
        "raw_output": metadata["raw_output"],
        "latency_sec": metadata["latency_sec"],
        "error": metadata["error"],
        "success": metadata["error"] is None
    }

In [8]:
from deepeval.test_case import LLMTestCase
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TimeElapsedColumn
from rich.console import Console

console = Console()

METRICS = [
    JSONValidityMetric(),
    SchemaComplianceMetric(),
    EntityAccuracyMetric(),
    MedicationRecallMetric(),
]

# results[model][example_id] = {metric_name: score, latency, error, ...}
results = {}

AVAILABLE_MODELS = check_models_available(MODELS_TO_EVAL)

print(AVAILABLE_MODELS)

total_calls = len(AVAILABLE_MODELS) * len(EXAMPLES)
console.print(f"\n[bold cyan]Starting evaluation: {len(AVAILABLE_MODELS)} models × {len(EXAMPLES)} examples = {total_calls} inference calls[/]\n")

with Progress(
    SpinnerColumn(),
    TextColumn("[bold blue]{task.description}"),
    BarColumn(),
    TextColumn("{task.completed}/{task.total}"),
    TimeElapsedColumn(),
    console=console
) as progress:
    task = progress.add_task("Evaluating...", total=total_calls)

    for model in AVAILABLE_MODELS:
        results[model] = {}
        for ex in EXAMPLES:
            progress.update(task, description=f"{model}")

            # 1. Inference
            infer = call_ollama(model, ex["data"]["ocr_text"], prompt, save_output=True)

            # 2. Build test case
            actual = infer["raw_output"] if infer["parsed_json"] is None else json.dumps(infer["parsed_json"])
            test_case = LLMTestCase(
                input=USER_TEMPLATE.format(ocr_text=ex["data"]["ocr_text"]),
                actual_output=actual,
                expected_output=json.dumps(ex["data"]["ground_truth"]),
            )

            # 3. Score each metric
            metric_scores = {}
            for metric in METRICS:
                metric.measure(test_case)
                metric_scores[metric.name] = round(metric.score, 4)

            results[model][ex["data"]["id"]] = {
                "scores": metric_scores,
                "latency": round(infer["latency_sec"], 2),
                "error": infer["error"],
                "parsed": infer["parsed_json"],
                "raw": infer["raw_output"][:300],  # Truncate for storage
            }
            progress.advance(task)

console.print("\n[bold green]✅ Evaluation complete![/]")

In [ ]:
import pandas as pd

METRIC_NAMES = [m.name for m in METRICS]

rows = []
for model in AVAILABLE_MODELS:
    model_results = results[model]
    n = len(model_results)
    avg_scores = {mn: 0.0 for mn in METRIC_NAMES}
    avg_latency = 0.0
    errors = 0

    for ex_id, res in model_results.items():
        for mn in METRIC_NAMES:
            avg_scores[mn] += res["scores"].get(mn, 0.0)
        avg_latency += res["latency"]
        if res["error"]:
            errors += 1

    row = {"Model": model}
    for mn in METRIC_NAMES:
        row[mn] = round(avg_scores[mn] / n, 3)
    row["Avg Latency (s)"] = round(avg_latency / n, 2)
   # row["Errors"] = errors
    # Overall score = mean of all metrics (excluding latency/errors)
    row["Overall"] = round(sum(row[mn] for mn in METRIC_NAMES) / len(METRIC_NAMES), 3)
    rows.append(row)

df = pd.DataFrame(rows).sort_values("Overall", ascending=False).reset_index(drop=True)
df.index += 1  # 1-based ranking
df.index.name = "Rank"

# Colour-coded display
def highlight(val, threshold=0.7):
    if isinstance(val, float):
        if val >= 0.9: color = "background-color: #2d6a4f; color: white"
        elif val >= 0.7: color = "background-color: #52b788"
        elif val >= 0.5: color = "background-color: #f4a261"
        else: color = "background-color: #e63946; color: white"
        return color
    return ""

styled = df.style.applymap(highlight, subset=METRIC_NAMES + ["Overall"])
styled

In [ ]:
# Pivot: for a chosen metric, show all models × all examples
TARGET_METRIC = "Entity Accuracy"  # ← Change to any metric name

pivot_rows = []
for model in AVAILABLE_MODELS:
    row = {"Model": model}
    for ex in EXAMPLES:
        score = results[model][ex["data"]["id"]]["scores"].get(TARGET_METRIC, None)
        row[ex["data"]["id"]] = round(score, 3) if score is not None else "ERR"
    pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).set_index("Model")
print(f"\n📊 {TARGET_METRIC} — per-example scores:")
pivot_df.style.applymap(highlight, subset=list(pivot_df.columns))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Plot 1: Overall score bar chart ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.RdYlGn(df["Overall"].values)
bars = axes[0].barh(df["Model"], df["Overall"], color=colors, edgecolor="white", height=0.6)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Score", fontsize=11)
axes[0].set_title("Overall Score (avg of all metrics)", fontsize=13, fontweight="bold")
axes[0].axvline(0.7, color="orange", linestyle="--", linewidth=1, label="threshold 0.7")
axes[0].legend()
for bar, val in zip(bars, df["Overall"]):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.2f}", va="center", fontsize=9)

# ── Plot 2: Latency vs Entity Accuracy scatter ─────────────────
axes[1].scatter(df["Avg Latency (s)"], df["Entity Accuracy"],
                c=df["Overall"], cmap="RdYlGn", s=120, edgecolors="black", linewidths=0.8)
for _, row_ in df.iterrows():
    axes[1].annotate(row_["Model"].split(":")[0],
                     (row_["Avg Latency (s)"], row_["Entity Accuracy"]),
                     textcoords="offset points", xytext=(5, 5), fontsize=8)
axes[1].set_xlabel("Avg Latency (s)", fontsize=11)
axes[1].set_ylabel("Entity Accuracy", fontsize=11)
axes[1].set_title("Quality vs Speed Trade-off", fontsize=13, fontweight="bold")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("/tmp/eval_plots1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 3: Radar / spider chart per model ─────────────────────
from matplotlib.patches import FancyArrowPatch

RADAR_METRICS = METRIC_NAMES  # all 4 metrics
N = len(RADAR_METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
cmap = plt.cm.tab10

for i, model in enumerate(AVAILABLE_MODELS):
    vals = [df[df["Model"] == model][mn].values[0] for mn in RADAR_METRICS]
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=1.8, color=cmap(i), label=model)
    ax.fill(angles, vals, alpha=0.08, color=cmap(i))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_METRICS, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_title("Model Capability Radar", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/eval_plots2.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Summary CSV ────────────────────────────────────────────────
df.to_csv("ocr_eval_summary.csv")
print("Saved: ocr_eval_summary.csv")

# ── Detailed CSV (every model × example × metric) ─────────────
detail_rows = []
for model in AVAILABLE_MODELS:
    for ex in EXAMPLES:
        r = results[model][ex["data"]["id"]]
        row = {"model": model, "example_id": ex["data"]["id"],
               "latency_s": r["latency"]}
        row.update(r["scores"])
        detail_rows.append(row)

detail_df = pd.DataFrame(detail_rows)
detail_df.to_csv("ocr_eval_detailed.csv", index=False)
print("Saved: ocr_eval_detailed.csv")

# ── Print final ranking ────────────────────────────────────────
console.print("\n[bold cyan]🏆 Final Ranking[/]")
for rank, (_, row_) in enumerate(df.iterrows(), 1):
    medal = ["🥇", "🥈", "🥉"].get(rank - 1, "  ")
    console.print(f"{medal} #{rank:2d}  [bold]{row_['Model']:<30}[/]  Overall: {row_['Overall']:.3f}  "
                  f"Entity Accuracy: {row_['Entity Accuracy']:.3f}  "
                  f"Latency: {row_['Avg Latency (s)']}s")